# Andean Obsidian Geochemical Sourcing

Welcome to the geochemical analysis notebook for XRF sourcing of obsidian from the Andes of South America.
This notebook provides the code for displaying geochemistry data together with obsidian source data from the region. It is intended to be reproducible and to provide a model for use with other data or analyses.

For long-term archival use, this notebook is paired with dependency files (`environment.yml`, `requirements.txt`) in the repository root that specify minimum compatible versions, so a working Python environment can be resolved in the future on whatever kernel/OS is available (local Miniconda, WSL, Binder, etc.).

Run all the cells before uploading files or clicking any buttons — use the menu **Run > Run All Cells** (in Voilà/Binder this happens automatically). The three tabs below walk through the workflow in order:

1. **Data Upload** — provide the three source tables (upload CSVs, or point to a Google Sheet / direct CSV URL).
2. **Map Selection** — the source chemistry and source-location tables are joined by `Group`, and you lasso-select obsidian sources of interest on a map.
3. **Biplot & Ternary** — biplot and ternary plots update automatically from your map selection so you can compare study samples against known obsidian sources.

The Python can be modified and re-run, and maps and plots are interactive and can be saved.


In [1]:
# CELL 1 — Imports
# Environment bootstrap for local, Jupyter, and Voila usage.
# For archival reproducibility, prefer the environment.yml / requirements.txt
# files in the repository root (minimum-version constraints, not exact pins,
# so they resolve on whatever kernel/OS is available).

from pathlib import Path
import os
import re
import sys
import io
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from types import SimpleNamespace

_is_binder = bool(os.environ.get("BINDER_LAUNCH_HOST"))
_is_voila = os.environ.get("SERVER_SOFTWARE", "").startswith("voila")

# Only attempt an on-the-fly pip install when running as a plain local kernel
# (not Binder/Voila, which already build their environment from environment.yml
# / requirements.txt ahead of time) AND only if the required packages are not
# already importable. This avoids re-installing/rebuilding packages (e.g.
# compiling numpy from source when no compiler is present) every time the
# notebook is run inside an already-working environment.
def _required_packages_importable():
    try:
        import numpy, pandas, plotly, ipywidgets  # noqa: F401
        return True
    except ImportError:
        return False

if not _is_binder and not _is_voila:
    if _required_packages_importable():
        print("Required packages already importable; skipping pip install.")
        print("If you have not done so, create the environment with:")
        print("  conda env create -f environment.yml   (conda/miniconda)")
        print("  python -m pip install -r requirements.txt   (pip/venv)")
    else:
        req_path = Path.cwd() / "requirements.txt"
        if not req_path.exists():
            req_path = Path.cwd().parent / "requirements.txt"

        if req_path.exists():
            print(f"Using dependency file: {req_path}")
            print("Installing dependencies from requirements.txt...")
            try:
                %pip install -q -r "{req_path}"
            except Exception as exc:
                print(f"⚠️ Automatic pip install failed: {exc}")
                print("Create the environment manually instead:")
                print("  conda env create -f environment.yml   (conda/miniconda, recommended)")
                print("  python -m pip install -r requirements.txt   (pip/venv)")
        else:
            print("No requirements.txt found. Create the environment with: conda env create -f environment.yml")
elif _is_voila:
    print("Voila-managed environment detected; dependencies come from the pre-built environment.")
else:
    print("Binder-managed environment detected; dependencies come from environment.yml")



try:
    from plotly.graph_objects import FigureWidget
    _plotly_figure_widget_available = True
except Exception:
    FigureWidget = go.Figure
    _plotly_figure_widget_available = False

from plotly.colors import DEFAULT_PLOTLY_COLORS
from ipywidgets import Button, Output, VBox, HBox, Tab, SelectMultiple, Layout, FileUpload, Text
import ipywidgets as widgets
from IPython.display import display, HTML

# Detect environment
def detect_environment():
    if os.environ.get("BINDER_LAUNCH_HOST"):
        return "binder"
    if os.environ.get("SERVER_SOFTWARE", "").startswith("voila"):
        return "voila"
    if os.environ.get("VSCODE_PID") or os.environ.get("TERM_PROGRAM") == "vscode":
        return "vscode"
    return "jupyter"

ENV = detect_environment()
print(f"Environment : {ENV}")
print(f"Python      : {sys.version.split()[0]}")
print(f"FigureWidget: {'available' if _plotly_figure_widget_available else 'falling back to go.Figure'}")

# Scrollable output — only inject in environments that support it
if ENV in ("jupyter", "vscode", "binder", "voila"):
    display(HTML("""
        <style>
            .output_wrapper, .output { 
                max-height: 600px !important; 
                overflow-y: auto !important; 
            }
        </style>
    """))

# ── Shared state object ────────────────────────────────────────────────────
# SimpleNamespace allows attribute access (state.srcs).
# Every step below reads and writes to this object so data flows between
# the tabs without relying on manually re-running cells, which is required
# for unattended execution under Voila/Binder.
state = SimpleNamespace(
    srcs=pd.DataFrame(),          # cleaned source chemistry
    srcs_locs=pd.DataFrame(),     # source coordinates
    study=pd.DataFrame(),         # cleaned study sample chemistry
    srcs_subset=pd.DataFrame(),   # sources filtered to the map selection (>=2 samples)
    onesample=pd.DataFrame(),     # selected sources with < 2 samples
    selected_names=list(),
    selected_groups=list(),
    name_to_color=dict(),
)

# Raw (pre-cleaning) dataframes, populated from default local files, uploads,
# or URLs (see Tab 1 below), and only cleaned/joined once "Load & Continue" runs.
_raw = SimpleNamespace(srcs=pd.DataFrame(), srcs_locs=pd.DataFrame(), study=pd.DataFrame())


Required packages already importable; skipping pip install.
If you have not done so, create the environment with:
  conda env create -f environment.yml   (conda/miniconda)
  python -m pip install -r requirements.txt   (pip/venv)
Environment : vscode
Python      : 3.13.14
FigureWidget: available


In [2]:
# CELL 2 - FLEXIBLE DATA LOADING HELPERS

from urllib.request import Request, urlopen
from urllib.parse import urlparse


SUPPORTED_EXTENSIONS = {
    ".csv",
    ".tsv",
    ".tab",
    ".txt",
    ".xlsx",
    ".xls",
}


def _source_extension(filename_or_url):
    """Return a lower-case extension, ignoring URL query strings."""
    value = str(filename_or_url or "")
    path = urlparse(value).path if "://" in value else value
    return Path(path).suffix.lower()


def _download_url(url):
    """Download a URL and return bytes plus the response content type."""
    request = Request(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
    )

    with urlopen(request, timeout=60) as response:
        return response.read(), response.headers.get("Content-Type", "")


def _looks_like_excel(raw_bytes):
    """Detect XLSX/XLS files from their binary signatures."""
    return (
        raw_bytes.startswith(b"PK")          # XLSX/ZIP container
        or raw_bytes.startswith(b"\xD0\xCF\x11\xE0")  # legacy XLS
    )


def _guess_separator(raw_bytes):
    """Infer a delimiter from the first usable text line."""
    text = raw_bytes.decode("utf-8-sig", errors="replace")

    lines = [
        line for line in text.splitlines()
        if line.strip()
    ]

    if not lines:
        return ","

    first_line = lines[0]

    candidates = {
        ",": first_line.count(","),
        "\t": first_line.count("\t"),
        ";": first_line.count(";"),
        "|": first_line.count("|"),
    }

    return max(candidates, key=candidates.get)


def read_data_table(content, filename_or_url=""):
    """
    Read a local path, URL, uploaded bytes, CSV, TSV, TXT, XLSX, or XLS file.

    The format is inferred from the filename/URL and, when necessary,
    from the downloaded/uploaded bytes.
    """

    source_name = str(filename_or_url or "")
    extension = _source_extension(source_name)

    # Resolve URLs.
    if isinstance(content, str) and content.startswith(("http://", "https://")):
        source_name = content
        extension = _source_extension(content)
        content, content_type = _download_url(content)
    else:
        content_type = ""

    # Resolve local paths.
    if isinstance(content, (str, Path)) and Path(content).exists():
        path = Path(content)
        extension = path.suffix.lower()

        if extension in {".xlsx", ".xls"}:
            return pd.read_excel(path)

        return pd.read_csv(
            path,
            sep=None,
            engine="python",
            encoding_errors="replace",
        )

    # Normalize uploaded memoryview/bytearray objects.
    if isinstance(content, memoryview):
        raw_bytes = content.tobytes()
    elif isinstance(content, bytearray):
        raw_bytes = bytes(content)
    elif isinstance(content, bytes):
        raw_bytes = content
    elif isinstance(content, str):
        raw_bytes = content.encode("utf-8")
    else:
        raise TypeError(f"Unsupported table input type: {type(content).__name__}")

    # Excel detection by extension or binary signature.
    if extension in {".xlsx", ".xls"} or _looks_like_excel(raw_bytes):
        return pd.read_excel(io.BytesIO(raw_bytes))

    # Text-table loading.
    separator = "\t" if extension in {".tsv", ".tab"} else _guess_separator(raw_bytes)

    return pd.read_csv(
        io.BytesIO(raw_bytes),
        sep=separator,
        engine="python",
        encoding="utf-8-sig",
        encoding_errors="replace",
    )


def gsheet_to_csv_url(url):
    """Convert a Google Sheets URL to a CSV export URL."""
    url = (url or "").strip()

    match = re.search(
        r"docs\.google\.com/spreadsheets/d/([a-zA-Z0-9_-]+)",
        url,
    )

    if not match:
        return url

    sheet_id = match.group(1)
    gid_match = re.search(r"(?:[?&#])gid=(\d+)", url)
    gid = gid_match.group(1) if gid_match else "0"

    return (
        f"https://docs.google.com/spreadsheets/d/{sheet_id}"
        f"/export?format=csv&gid={gid}"
    )


def _first_upload(upload_widget):
    """Support ipywidgets 7 and 8 upload-value formats."""
    value = upload_widget.value

    if isinstance(value, dict):
        return next(iter(value.values()), None)

    if isinstance(value, (list, tuple)):
        return value[0] if value else None

    return None

In [ ]:
# CELL 3 - DATA CLEANING, VALIDATION AND JOIN

def clean_geochem_df(df):
    """Clean geochemistry dataframe headers and data types."""
    if df is None or df.empty:
        return pd.DataFrame()

    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(r"(Ka1|La1|\s+)", "", regex=True)
    )

    string_cols = ["Group", "Sample", "Name"]

    for column in string_cols:
        if column in df.columns:
            df[column] = df[column].astype("string").str.strip()

    numeric_cols = [
        column for column in df.columns
        if column not in string_cols
    ]

    for column in numeric_cols:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    return df.dropna(axis=1, how="all")


def _remove_duplicate_columns(df):
    """Drop repeated column names, keeping the first occurrence."""
    if not isinstance(df, pd.DataFrame):
        return pd.DataFrame()

    result = df.copy()
    result.columns = result.columns.astype(str).str.strip()

    return result.loc[
        :,
        ~result.columns.duplicated(keep="first"),
    ]


# Group is optional for the study table: study samples may not yet be
# assigned to a known source group.
REQUIRED = {
    "srcs": ["Sample", "Group", "Rb", "Sr", "Zr"],
    "study": ["Sample", "Rb", "Sr", "Zr"],
}


def check_schema(df, required, name):
    """Report missing required columns."""
    missing = [
        column for column in required
        if column not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{name} missing columns: {missing}. "
            f"Available columns: {list(df.columns)}"
        )


def join_sources_locations():
    """Join source chemistry to source coordinates and report coverage."""
    srcs_locs = state.srcs_locs

    if srcs_locs is None or srcs_locs.empty:
        print("⚠️ Source locations not loaded.")
        return

    # Chem_Group is the join key shared by srcs and srcs_locs; it is
    # distinct from either table's own Group column.
    required_location_columns = [
        "Chem_Group",
        "Lat",
        "Long",
        "Name",
    ]

    missing = [
        column
        for column in required_location_columns
        if column not in srcs_locs.columns
    ]

    if missing:
        print(f"⚠️ Source locations missing columns: {missing}")
        print(
            "Available columns: "
            f"{srcs_locs.columns.tolist()}"
        )
        return

    if "Chem_Group" not in state.srcs.columns:
        print(
            "⚠️ Sources table has no 'Chem_Group' column; skipping join to "
            f"locations. Available columns: {list(state.srcs.columns)}"
        )
        return

    location_table = (
        srcs_locs[required_location_columns]
        .copy()
        .drop_duplicates(subset=["Chem_Group"])
    )

    location_table["Lat"] = pd.to_numeric(
        location_table["Lat"],
        errors="coerce",
    )

    location_table["Long"] = pd.to_numeric(
        location_table["Long"],
        errors="coerce",
    )

    location_table = location_table.rename(
        columns={
            "Lat": "Lat_loc",
            "Long": "Long_loc",
        }
    )

    joined_srcs = state.srcs.merge(
        location_table,
        on="Chem_Group",
        how="left",
    )

    print("✅ Joined source chemistry to source locations.")
    print(f"Rows: {len(joined_srcs)}")

    missing_coordinates = joined_srcs[
        ["Lat_loc", "Long_loc"]
    ].isna().any(axis=1).sum()

    if missing_coordinates:
        print(
            f"⚠️ {missing_coordinates} joined row(s) "
            "have missing coordinates."
        )
    else:
        print("✅ All joined rows have coordinates.")


def clean_and_validate(raw_srcs, raw_locs, raw_study):
    """Clean, standardize, validate, and join the uploaded tables."""

    # Apply column aliases before cleaning.
    state.srcs = _standardize_table_columns(
        raw_srcs,
        "sources",
    )

    state.srcs_locs = _standardize_table_columns(
        raw_locs,
        "locations",
    )

    state.study = _standardize_table_columns(
        raw_study,
        "study",
    )

    # Clean table contents and remove duplicate columns.
    state.srcs = _remove_duplicate_columns(
        clean_geochem_df(state.srcs)
    )

    state.study = _remove_duplicate_columns(
        clean_geochem_df(state.study)
    )

    state.srcs_locs = _remove_duplicate_columns(
        state.srcs_locs
    )

    # Join chemistry to source locations on Chem_Group and prefer the
    # location table's Group field as the canonical source category.
    if (
        isinstance(state.srcs, pd.DataFrame)
        and not state.srcs.empty
        and isinstance(state.srcs_locs, pd.DataFrame)
        and not state.srcs_locs.empty
        and "Chem_Group" in state.srcs.columns
        and "Chem_Group" in state.srcs_locs.columns
        and "Group" in state.srcs_locs.columns
    ):
        loc_group = state.srcs_locs[
            ["Chem_Group", "Group"]
        ].drop_duplicates(subset=["Chem_Group"]).copy()
        loc_group = loc_group.rename(columns={"Group": "Group_loc"})

        state.srcs = state.srcs.merge(
            loc_group,
            on="Chem_Group",
            how="left",
        )

        if "Group" in state.srcs.columns:
            state.srcs["Group"] = state.srcs["Group_loc"].fillna(
                state.srcs["Group"]
            )
        else:
            state.srcs["Group"] = state.srcs["Group_loc"]

        state.srcs = state.srcs.drop(columns=["Group_loc"])

    # Validate source and study chemistry tables.
    for table_name, table, required_columns in [
        ("Sources", state.srcs, REQUIRED["srcs"]),
        ("Study", state.study, REQUIRED["study"]),
    ]:
        if table is None or table.empty:
            print(f"⚠️ {table_name} table is empty.")
            continue

        try:
            check_schema(
                table,
                required_columns,
                table_name,
            )
        except ValueError as exc:
            print(f"⚠️ {exc}")

    # Display the requested source summary table.
    print("Source groups used in this visualization:")

    if "Group" not in state.srcs.columns:
        print(
            "⚠️ Sources table has no 'Group' column; skipping source "
            f"summary. Available columns: {list(state.srcs.columns)}"
        )
    else:
        source_counts = (
            state.srcs
            .groupby("Group", dropna=False)
            .size()
            .reset_index(name="Count")
            .sort_values(
                "Count",
                ascending=False,
            )
        )

        if "Group" not in state.srcs_locs.columns:
            print(
                "⚠️ Source-location table has no 'Group' column; "
                "showing counts without location names."
            )
            source_summary = source_counts.copy()
            source_summary["Source"] = source_summary["Group"]
        else:
            location_names = (
                state.srcs_locs[
                    ["Group", "Name"]
                ]
                .drop_duplicates(subset=["Group"])
                if "Name" in state.srcs_locs.columns
                else state.srcs_locs[
                    ["Group"]
                ].drop_duplicates()
            )

            source_summary = location_names.merge(
                source_counts,

                on="Group",        display(source_summary)

                how="left",

            )        join_sources_locations()



            source_summary["Count"] = (        ].reset_index(drop=True)

                source_summary["Count"].fillna(0).astype(int)            ["Source", "Group", "Count"]

            )        source_summary = source_summary[



            if "Name" in source_summary.columns:            source_summary = source_summary.drop_duplicates(subset=["Group"]).reset_index(drop=True)

                source_summary = source_summary.rename(        if "Group" in state.srcs_locs.columns and "Group" in source_summary.columns:

                    columns={"Name": "Source"}

                )            ].sort_values("Count", ascending=False).reset_index(drop=True)

            else:                ["Source", "Group", "Count"]

                source_summary["Source"] = (            source_summary = source_summary[

                    source_summary["Group"]
                )

In [4]:
# CELL 4 - SOURCE SELECTION WITH FIGUREWIDGET

def _show_status_popup(output, title, message, kind="info"):
    """Display a dismissible status popup."""
    colors = {
        "success": ("#e8f5e9", "#2e7d32"),
        "warning": ("#fff3e0", "#ef6c00"),
        "error": ("#ffebee", "#c62828"),
        "info": ("#e3f2fd", "#1565c0"),
    }

    background, border = colors.get(kind, colors["info"])

    close_button = widgets.Button(
        description="Close",
        layout=widgets.Layout(width="80px"),
    )

    popup = widgets.VBox(
        [
            widgets.HTML(
                value=(
                    f"<b style='font-size:16px;'>{title}</b>"
                    f"<div style='margin-top:8px; white-space:pre-wrap;'>"
                    f"{message}</div>"
                )
            ),
            close_button,
        ],
        layout=widgets.Layout(
            position="fixed",
            top="15%",
            left="50%",
            width="420px",
            padding="18px",
            z_index="9999",
            background=background,
            border=f"3px solid {border}",
            border_radius="8px",
            box_shadow="0 4px 16px rgba(0,0,0,0.3)",
        ),
    )

    def close_popup(_):
        popup.close()

    close_button.on_click(close_popup)

    with output:
        output.clear_output(wait=True)
        display(popup)


def _patched_delta_handler(fig_instance):
    """
    Protect FigureWidget from invalid Plotly trace and relayout updates.
    """
    original_delta_handler = getattr(
        fig_instance,
        "_handler_js2py_traceDeltas",
        None,
    )

    if original_delta_handler is not None:

        def safe_delta_handler(change):
            try:
                new_value = change.get("new")

                if isinstance(new_value, list):
                    valid_deltas = [
                        delta
                        for delta in new_value
                        if isinstance(delta, dict)
                        and "uid" in delta
                    ]

                    if not valid_deltas:
                        return

                    change_copy = dict(change)
                    change_copy["new"] = valid_deltas

                    try:
                        original_delta_handler(change_copy)
                    except (KeyError, IndexError):
                        pass
                else:
                    original_delta_handler(change)

            except Exception:
                pass

        fig_instance._handler_js2py_traceDeltas = safe_delta_handler

        trace_store = getattr(
            fig_instance,
            "_traceDeltas",
            None,
        )

        if (
            trace_store is not None
            and hasattr(trace_store, "unobserve")
            and hasattr(trace_store, "observe")
        ):
            try:
                trace_store.unobserve(
                    original_delta_handler,
                    names="value",
                )
            except Exception:
                pass

            try:
                trace_store.observe(
                    safe_delta_handler,
                    names="value",
                )
            except Exception:
                pass

    original_relayout_handler = getattr(
        fig_instance,
        "_handler_js2py_relayout",
        None,
    )

    if original_relayout_handler is not None:

        def safe_relayout_handler(change):
            try:
                relayout_data = change.get("new", {})

                if isinstance(relayout_data, dict):
                    cleaned_relayout = {
                        key: value
                        for key, value in relayout_data.items()
                        if "_derived" not in key
                    }

                    if cleaned_relayout:
                        change_copy = dict(change)
                        change_copy["new"] = cleaned_relayout
                        original_relayout_handler(change_copy)
                else:
                    original_relayout_handler(change)

            except ValueError as exc:
                if "Invalid property path" not in str(exc):
                    raise
            except Exception:
                pass

        fig_instance._handler_js2py_relayout = safe_relayout_handler

        relayout_store = getattr(
            fig_instance,
            "_js2py_relayout",
            None,
        )

        if (
            relayout_store is not None
            and hasattr(relayout_store, "unobserve")
            and hasattr(relayout_store, "observe")
        ):
            try:
                relayout_store.unobserve(
                    original_relayout_handler,
                    names="value",
                )
            except Exception:
                pass

            try:
                relayout_store.observe(
                    safe_relayout_handler,
                    names="value",
                )
            except Exception:
                pass


def build_map_selection_ui(host_out):
    """Render the source map and lasso-selection controls."""

    with host_out:
        host_out.clear_output(wait=True)

        display(widgets.HTML(
            """
            <h3>Joined South American obsidian source chemistry with coordinates</h3>
            <p>
            The source chemistry and source-location tables are joined by
            <code>Chem_Group</code>, which appears in both tables.
            Use the lasso tool to select sources, then click
            <b>Get Selection</b>.
            </p>
            """
        ))

        srcs_locs = state.srcs_locs.copy()

        # Chem_Group is the join key shared by srcs and srcs_locs; it is
        # distinct from either table's own Group column.
        required_columns = ["Name", "Chem_Group", "Lat", "Long"]
        missing_columns = [
            column
            for column in required_columns
            if column not in srcs_locs.columns
        ]

        if missing_columns:
            print(
                "⚠️ Source-location table is missing: "
                f"{missing_columns}"
            )
            return

        srcs_locs["Lat"] = pd.to_numeric(
            srcs_locs["Lat"],
            errors="coerce",
        )

        srcs_locs["Long"] = pd.to_numeric(
            srcs_locs["Long"],
            errors="coerce",
        )

        srcs_locs = srcs_locs.dropna(
            subset=["Lat", "Long"]
        )

        if srcs_locs.empty:
            print(
                "⚠️ Source-location table contains no valid coordinates."
            )
            return
        groups = srcs_locs["Chem_Group"].astype(str).tolist()
        names = srcs_locs["Name"].astype(str).tolist()
        lats = srcs_locs["Lat"].astype(float).tolist()
        lons = srcs_locs["Long"].astype(float).tolist()

        joined_groups_set = (
            set(
                state.srcs["Chem_Group"]
                .dropna()
                .astype(str)
            )
            if (
                isinstance(state.srcs, pd.DataFrame)
                and not state.srcs.empty
                and "Chem_Group" in state.srcs.columns
            )
            else set()
        )

        joined_idx = [
            index
            for index, group in enumerate(groups)
            if group in joined_groups_set
        ]

        unjoined_idx = [
            index
            for index, group in enumerate(groups)
            if group not in joined_groups_set
        ]

        joined_names = [
            names[index]
            for index in joined_idx
        ]

        joined_groups_list = [
            groups[index]
            for index in joined_idx
        ]

        unjoined_names = [
            names[index]
            for index in unjoined_idx
        ]

        unjoined_groups_list = [
            groups[index]
            for index in unjoined_idx
        ]

        center_lats = [
            lats[index]
            for index in joined_idx
        ] or lats

        center_lons = [
            lons[index]
            for index in joined_idx
        ] or lons

        center_lat = np.mean(center_lats)
        center_lon = np.mean(center_lons)

        max_span = (
            max(
                max(center_lats) - min(center_lats),
                max(center_lons) - min(center_lons),
            )
            if len(center_lats) > 1
            else 1
        )

        zoom = (
            5 if max_span > 10 else
            6 if max_span > 5 else
            7 if max_span > 2 else
            8 if max_span > 1 else
            9 if max_span > 0.5 else
            11
        )

        trace_joined = go.Scattermapbox(
            lat=[
                lats[index]
                for index in joined_idx
            ],
            lon=[
                lons[index]
                for index in joined_idx
            ],
            mode="markers+text",
            text=joined_names,
            textposition="top center",
            customdata=joined_groups_list,
            marker=dict(
                size=10,
                color="steelblue",
                symbol="circle",
            ),
            selected=dict(
                marker=dict(
                    size=14,
                    color="red",
                )
            ),
            unselected=dict(
                marker=dict(opacity=0.4)
            ),
            hovertemplate=(
                "<b>%{text}</b><br>"
                "<b>Group:</b> %{customdata}<br>"
                "Has chemistry data<extra></extra>"
            ),
            name="Sources with chemistry data",
        )

        trace_joined.uid = "scattermap_joined"

        trace_unjoined = go.Scattermapbox(
            lat=[
                lats[index]
                for index in unjoined_idx
            ],
            lon=[
                lons[index]
                for index in unjoined_idx
            ],
            mode="markers+text",
            text=unjoined_names,
            textposition="top center",
            textfont=dict(color="lightgray"),
            customdata=unjoined_groups_list,
            marker=dict(
                size=10,
                color="black",
                symbol="x",
            ),
            selected=dict(
                marker=dict(
                    size=14,
                    color="red",
                )
            ),
            unselected=dict(
                marker=dict(opacity=0.4)
            ),
            hovertemplate=(
                "<b>%{text}</b><br>"
                "<b>Group:</b> %{customdata}<br>"
                "No chemistry data<extra></extra>"
            ),
            name="Sources without chemistry data",
        )

        trace_unjoined.uid = "scattermap_unjoined"

        fig = FigureWidget(
            data=[
                trace_joined,
                trace_unjoined,
            ]
        )

        _patched_delta_handler(fig)

        fig.update_layout(
            mapbox=dict(
                style="open-street-map",
                center=dict(
                    lat=center_lat,
                    lon=center_lon,
                ),
                zoom=zoom,
            ),
            dragmode="lasso",
            height=500,
            margin=dict(
                r=0,
                l=0,
                t=30,
                b=0,
            ),
            title=(
                "🗺️ Obsidian Source Locations — "
                "lasso to select, then click Get Selection"
            ),
            hovermode="closest",
            showlegend=True,
        )

        display(fig)

        selection_out = widgets.Output()

        button = widgets.Button(
            description="Get Selection",
            button_style="primary",
            icon="check",
            layout=widgets.Layout(width="180px"),
        )

        def on_get_selection(_):
            try:
                selected_joined = (
                    list(fig.data[0].selectedpoints or [])
                    if len(fig.data) > 0
                    else []
                )

                selected_unjoined = (
                    list(fig.data[1].selectedpoints or [])
                    if len(fig.data) > 1
                    else []
                )

                if not selected_joined and not selected_unjoined:
                    _show_status_popup(
                        selection_out,
                        "No points selected",
                        "Draw a lasso around one or more "
                        "sources on the map first.",
                        kind="warning",
                    )
                    return

                selected_pairs = (
                    [
                        (
                            joined_names[index],
                            joined_groups_list[index],
                        )
                        for index in selected_joined
                    ]
                    + [
                        (
                            unjoined_names[index],
                            unjoined_groups_list[index],
                        )
                        for index in selected_unjoined
                    ]
                )

                seen_groups = set()
                unique_pairs = []

                for name, group in selected_pairs:
                    if group not in seen_groups:
                        seen_groups.add(group)
                        unique_pairs.append((name, group))

                state.selected_names = [
                    name
                    for name, group in unique_pairs
                ]

                state.selected_groups = [
                    group
                    for name, group in unique_pairs
                ]

                _show_status_popup(
                    selection_out,
                    "Selection successful",
                    (
                        f"{len(state.selected_groups)} source(s) selected:\n\n"
                        + "\n".join(
                            f"• {group}"
                            for group in state.selected_groups
                        )
                    ),
                    kind="success",
                )

            except Exception as exc:
                _show_status_popup(
                    selection_out,
                    "Selection error",
                    str(exc),
                    kind="error",
                )
                return

            render_analysis(tab3_out)
            tabs.selected_index = 2

        button.on_click(on_get_selection)

        display(widgets.HTML(
            "<b>After lasso-selecting sources, click:</b>"
        ))

        display(
            widgets.VBox(
                [
                    button,
                    selection_out,
                ]
            )
        )


def render_selection_summary():
    """Display the selected-source summary."""

    if not state.selected_names:
        display(HTML(
            """
            <div style="
                padding:10px;
                background-color:#fff3e0;
                border-radius:5px;
                border-left:4px solid #FF9800;
            ">
                <b>⚠️ No selections yet.</b>
                Select sources on the map first.
            </div>
            """
        ))
        return

    try:
        summary = []

        for name, group in zip(
            state.selected_names,
            state.selected_groups,
        ):
            if group in state.srcs["Group"].values:
                count = len(
                    state.srcs[
                        state.srcs["Group"] == group
                    ]
                )

                summary.append(
                    {
                        "Source": name,
                        "Group": group,
                        "Count": count,
                    }
                )

        if summary:
            summary_df = (
                pd.DataFrame(summary)
                .sort_values(
                    "Count",
                    ascending=False,
                )
                .reset_index(drop=True)
            )

            display(summary_df)

        else:
            display(HTML(
                """
                <div style="
                    padding:10px;
                    background-color:#ffebee;
                    border-radius:5px;
                    border-left:4px solid #f44336;
                ">
                    <b>❌ No selected sources found in the chemistry data.</b>
                </div>
                """
            ))

    except Exception as exc:
        display(HTML(
            f"""
            <div style="
                padding:10px;
                background-color:#ffebee;
                border-radius:5px;
                border-left:4px solid #f44336;
            ">
                <b>❌ Error processing selection:</b>
                {exc}
            </div>
            """

        ))

In [ ]:
# CELL 5 - APPLY MAP SELECTION AND BUILD COLOR MAP
# Invoked by render_analysis() once a map selection has been applied.

def apply_selection():
    if not state.selected_groups:
        print("❌ Error: No sources selected. Please select sources from the map first.")
        state.srcs_subset = pd.DataFrame()
        state.onesample = pd.DataFrame()
        return

    try:
        if 'Chem_Group' not in state.srcs.columns:
            raise KeyError(
                f"srcs table missing 'Chem_Group' column needed to match the "
                f"map selection. Available columns: {list(state.srcs.columns)}"
            )

        srcs_subset = state.srcs[state.srcs['Chem_Group'].isin(state.selected_groups)].copy()

        # Prepare a normalized locations table from srcs_locs (handle varied column names)
        def _find_col(df, names):
            names_low = [n.lower() for n in names]
            for c in df.columns:
                if c.lower() in names_low:
                    return c
            return None

        srcs_locs = state.srcs_locs
        # Chem_Group is the join key shared by srcs and srcs_locs; srcs_locs'
        # own Group column (if present) is a different field and must not be used.
        group_col = _find_col(srcs_locs, ['Chem_Group', 'chem_group']) or 'Chem_Group'
        lat_col = _find_col(srcs_locs, ['Lat', 'Latitude', 'lat', 'latitude', 'Lat_loc'])
        long_col = _find_col(srcs_locs, ['Long', 'Longitude', 'Long_loc', 'Lon', 'lon', 'longitude'])
        name_col = _find_col(srcs_locs, ['Name', 'name', 'Source', 'source'])

        if lat_col is None or long_col is None:
            raise KeyError(f"srcs_locs missing coordinate columns. Available columns: {list(srcs_locs.columns)}")

        loc_rename = {lat_col: 'Lat', long_col: 'Long'}
        if group_col != 'Chem_Group':
            loc_rename[group_col] = 'Chem_Group'
        if name_col:
            loc_rename[name_col] = 'Name'
        srcs_locs_norm = srcs_locs.rename(columns=loc_rename)

        if 'Chem_Group' not in srcs_locs_norm.columns:
            raise KeyError(f"Could not locate a 'Chem_Group' column in srcs_locs. Available: {list(srcs_locs.columns)}")

        srcs_locs_coords = srcs_locs_norm[['Chem_Group', 'Lat', 'Long'] + (['Name'] if 'Name' in srcs_locs_norm.columns else [])]
        srcs_locs_coords = srcs_locs_coords.drop_duplicates(subset=['Chem_Group']).reset_index(drop=True)

        srcs_subset = srcs_subset.merge(srcs_locs_coords, on='Chem_Group', how='left')

        if len(srcs_subset) == 0:
            raise ValueError(f"No data found for selected sources: {state.selected_groups}")

        if 'Lat' not in srcs_subset.columns or 'Long' not in srcs_subset.columns:
            print(f"⚠️ After merge, location columns missing. Available columns: {list(srcs_subset.columns)}")
        elif srcs_subset[['Lat', 'Long']].isna().any().any():
            print("⚠️ Some selected sources have no matching location data.")

        print(f"✅ Filtered to {len(srcs_subset)} samples from {len(state.selected_groups)} selected sources")

        counts = srcs_subset['Group'].value_counts()
        onesample = srcs_subset[srcs_subset['Group'].map(counts) < 2]
        srcs_subset = srcs_subset[srcs_subset['Group'].map(counts) >= 2]

        print(f"✅ {len(srcs_subset)} samples for ellipses")
        print(f"✅ {len(onesample)} samples for points (< 2 per source)")

        state.srcs_subset = srcs_subset
        state.onesample = onesample
    except ValueError as e:
        print(f"❌ Error: {e}")
        state.srcs_subset = pd.DataFrame()
        state.onesample = pd.DataFrame()
    except KeyError as e:
        print(f"❌ Key error: {e}")
        state.srcs_subset = pd.DataFrame()
        state.onesample = pd.DataFrame()
    except Exception as e:
        print(f"❌ Unexpected error: {type(e).__name__}: {e}")
        state.srcs_subset = pd.DataFrame()
        state.onesample = pd.DataFrame()


def build_color_map():
    """Assign a color to each Source and each Study Group for consistency.
    Invoked by render_analysis() once the selection has been applied."""
    loc_group_values = (
        state.srcs_locs['Group'].dropna().unique()
        if isinstance(state.srcs_locs, pd.DataFrame)
        and 'Group' in state.srcs_locs.columns
        else np.array([], dtype=object)
    )


    src_group_values = (    state.name_to_color = {name: colors[i % len(colors)] for i, name in enumerate(all_groups)}

        state.srcs['Group'].dropna().unique()    # Cycle colors if more groups than colors

        if isinstance(state.srcs, pd.DataFrame)

        and 'Group' in state.srcs.columns    colors = DEFAULT_PLOTLY_COLORS

        else np.array([], dtype=object)    ) if len(loc_group_values) or len(src_group_values) or len(study_groups) else np.array([], dtype=object)

    )        ])

            np.asarray(study_groups, dtype=object),

    study_groups = (            np.asarray(src_group_values, dtype=object),

        state.study['Group'].dropna().unique()            np.asarray(loc_group_values, dtype=object),

        if isinstance(state.study, pd.DataFrame)        np.concatenate([

        and 'Group' in state.study.columns    all_groups = pd.unique(

        else np.array([], dtype=object)
    )

In [6]:
# CELL 6 - BIPLOT

def ellipse_points(x, y, n_std=1.0, n_points=120):
    """Return points tracing a covariance ellipse."""
    x = pd.to_numeric(pd.Series(x), errors="coerce").to_numpy(dtype=float)
    y = pd.to_numeric(pd.Series(y), errors="coerce").to_numpy(dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 2:
        return None

    covariance = np.cov(x, y)

    if not np.isfinite(covariance).all():
        return None

    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    eigenvalues = np.maximum(eigenvalues, 0)

    order = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    theta = np.linspace(0, 2 * np.pi, n_points)
    unit_circle = np.column_stack([
        np.cos(theta),
        np.sin(theta),
    ])

    ellipse = (
        unit_circle
        @ np.diag(n_std * np.sqrt(eigenvalues))
        @ eigenvectors.T
    )

    ellipse += [x.mean(), y.mean()]
    return ellipse


def build_biplot(x_col, y_col, show_labels=False, show_source_points=True):
    GROUP_COL = "Group"
    fig = go.Figure()

    srcs_subset_valid = (
        state.srcs_subset.copy()
        if (
            isinstance(state.srcs_subset, pd.DataFrame)
            and not state.srcs_subset.empty
        )
        else pd.DataFrame()
    )

    source_data = (
        srcs_subset_valid.copy()
        if (
            not srcs_subset_valid.empty
            and GROUP_COL in srcs_subset_valid.columns
        )
        else pd.DataFrame()
    )

    # Source ellipses and source points.
    if (
        not source_data.empty
        and x_col in source_data.columns
        and y_col in source_data.columns
    ):
        source_groups = (
            source_data[GROUP_COL]
            .dropna()
            .astype(str)
            .unique()
        )

        for source_group in source_groups:
            group_data = source_data[
                source_data[GROUP_COL].astype(str) == source_group
            ].copy()

            numeric_x = pd.to_numeric(group_data[x_col], errors="coerce")
            numeric_y = pd.to_numeric(group_data[y_col], errors="coerce")
            valid_points = numeric_x.notna() & numeric_y.notna()

            valid_group_data = group_data.loc[valid_points].copy()
            valid_x = numeric_x.loc[valid_points]
            valid_y = numeric_y.loc[valid_points]

            if len(valid_group_data) < 2:
                continue

            color = state.name_to_color.get(source_group, "gray")
            ellipse = ellipse_points(valid_x, valid_y, n_std=1.0)

            if ellipse is not None:
                fig.add_trace(go.Scatter(
                    x=ellipse[:, 0],
                    y=ellipse[:, 1],
                    mode="lines",
                    fill="toself",
                    fillcolor=color,
                    line=dict(color=color, width=2),
                    opacity=0.35,
                    name=f"{source_group} Source Ellipse",
                    hoverinfo="skip",
                ))

                fig.add_trace(go.Scatter(
                    x=[valid_x.mean()],
                    y=[valid_y.mean()],
                    mode="text",
                    text=[f"{source_group} Source"],
                    textposition="middle center",
                    textfont=dict(size=11, color=color),
                    showlegend=False,
                    hoverinfo="skip",
                ))

            if show_source_points:
                point_text = (
                    valid_group_data["Sample"]
                    if "Sample" in valid_group_data.columns
                    else valid_group_data["Name"]
                    if "Name" in valid_group_data.columns
                    else None
                )

                fig.add_trace(go.Scatter(
                    x=valid_x,
                    y=valid_y,
                    mode="markers",
                    name=f"{source_group} Source Points",
                    text=point_text,
                    hovertemplate="Source sample: %{text}<extra></extra>",
                    marker=dict(
                        size=6,
                        color=color,
                        opacity=0.8,
                        line=dict(width=1, color="black"),
                    ),
                    showlegend=False,
                ))

    # Source samples with fewer than two observations.
    onesample = state.onesample

    if (
        isinstance(onesample, pd.DataFrame)
        and not onesample.empty
        and x_col in onesample.columns
        and y_col in onesample.columns
    ):
        valid = (
            pd.to_numeric(onesample[x_col], errors="coerce").notna()
            & pd.to_numeric(onesample[y_col], errors="coerce").notna()
        )

        source_sample_text = (
            onesample.loc[valid, "Sample"]
            if "Sample" in onesample.columns
            else None
        )

        fig.add_trace(go.Scatter(
            x=pd.to_numeric(
                onesample.loc[valid, x_col],
                errors="coerce",
            ),
            y=pd.to_numeric(
                onesample.loc[valid, y_col],
                errors="coerce",
            ),
            name="Source Sample (<2)",
            mode="markers",
            marker=dict(symbol="x", size=8, color="black"),
            text=source_sample_text,
            hovertemplate="Source: %{text}<br><extra></extra>",
            showlegend=True,
        ))

    # Study samples.
    study = state.study

    if (
        isinstance(study, pd.DataFrame)
        and not study.empty
        and GROUP_COL in study.columns
        and x_col in study.columns
        and y_col in study.columns
    ):
        study_groups = (
            study[GROUP_COL]
            .dropna()
            .astype(str)
            .unique()
        )

        for study_group in study_groups:
            group_data = study[
                study[GROUP_COL].astype(str) == study_group
            ].copy()

            x_values = pd.to_numeric(group_data[x_col], errors="coerce")
            y_values = pd.to_numeric(group_data[y_col], errors="coerce")
            valid = x_values.notna() & y_values.notna()

            if not valid.any():
                continue

            color = state.name_to_color.get(study_group, "gray")

            sample_text = (
                group_data.loc[valid, "Sample"]
                if "Sample" in group_data.columns
                else group_data.loc[valid, "Name"]
                if "Name" in group_data.columns
                else None
            )

            fig.add_trace(go.Scatter(
                x=x_values.loc[valid],
                y=y_values.loc[valid],
                name=f"Study: {study_group}",
                mode="markers+text" if show_labels else "markers",
                text=sample_text,
                textposition="top center",
                hovertemplate="Sample: %{text}<br><extra></extra>",
                marker=dict(
                    size=8,
                    symbol="circle",
                    color=color,
                ),
                showlegend=True,
            ))

    fig.update_layout(
        title=dict(
            text="Connecting Study Samples with known obsidian sources",
            x=0.5,
            xanchor="center",
        ),
        xaxis_title=x_col,
        yaxis_title=y_col,
        height=600,
        margin=dict(l=0, r=240, t=120, b=0),
        legend=dict(
            x=1.02,
            y=0.98,
            xanchor="left",
            yanchor="top",
        ),
    )

    return fig


def render_biplot_ui(
    host_out,
    show_labels_getter=lambda: False,
    show_source_points_getter=lambda: True,
):
    """Render the biplot section."""
    state.srcs = _remove_duplicate_columns(state.srcs)
    state.study = _remove_duplicate_columns(state.study)
    state.srcs_subset = _remove_duplicate_columns(state.srcs_subset)

    plot_elements = ["Rb", "Sr", "Zr", "Ba", "Nb", "Y", "Th", "U"]
    available_elements = [
        column
        for column in plot_elements
        if column in state.srcs.columns
        and column in state.study.columns
    ]

    with host_out:
        display(HTML(
            "<h3>Biplot</h3>"
            "<p style='color:grey;font-size:12px;'>"
            "Change the element variables on the axes or use the shared "
            "controls above to show labels and source points."
            "</p>"
        ))

        if not available_elements:
            print(
                "⚠️ No common geochemical columns are available "
                "in srcs and study."
            )
            return

        x_default = "Sr" if "Sr" in available_elements else available_elements[0]
        y_default = "Rb" if "Rb" in available_elements else available_elements[0]

        x_menu = widgets.Dropdown(
            options=available_elements,
            value=x_default,
            description="X axis:",
            style={"description_width": "initial"},
        )

        y_menu = widgets.Dropdown(
            options=available_elements,
            value=y_default,
            description="Y axis:",
            style={"description_width": "initial"},
        )

        display(widgets.HBox([x_menu, y_menu]))

        biplot_output = widgets.Output()

        def update_biplot(change=None):
            with biplot_output:
                biplot_output.clear_output(wait=True)
                display(build_biplot(
                    x_menu.value,
                    y_menu.value,
                    show_labels=show_labels_getter(),
                    show_source_points=show_source_points_getter(),
                ))

        x_menu.observe(update_biplot, names="value")
        y_menu.observe(update_biplot, names="value")

        display(biplot_output)
        update_biplot()

In [7]:
# CELL 7 - TERNARY PLOT + ANALYSIS ORCHESTRATOR

def normalize_composition(df, cols):
    values = (
        df[cols]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype=float)
    )
    totals = np.sum(values, axis=1, keepdims=True)

    valid = np.isfinite(totals[:, 0]) & (totals[:, 0] > 0)
    result = np.full_like(values, np.nan)

    with np.errstate(divide="ignore", invalid="ignore"):
        result[valid] = values[valid] / totals[valid]

    return result


def build_ternary_plot(
    a_col,
    b_col,
    c_col,
    show_labels=False,
    show_source_points=True,
):
    GROUP_COL = "Group"
    cols = [a_col, b_col, c_col]
    fig = go.Figure()

    srcs_subset_valid = (
        state.srcs_subset.copy()
        if (
            isinstance(state.srcs_subset, pd.DataFrame)
            and not state.srcs_subset.empty
        )
        else pd.DataFrame()
    )

    study = state.study
    source_data = srcs_subset_valid.copy()

    source_fraction = (
        normalize_composition(source_data, cols)
        if not source_data.empty
        else None
    )

    study_fraction = (
        normalize_composition(study, cols)
        if not study.empty
        else None
    )

    # Source ellipses and source points.
    if (
        source_fraction is not None
        and not source_data.empty
        and GROUP_COL in source_data.columns
    ):
        for source_group in source_data[GROUP_COL].dropna().unique():
            group_mask = (
                source_data[GROUP_COL].astype(str)
                == str(source_group)
            ).to_numpy()

            group_data = source_data.loc[group_mask].copy()
            group_fraction = source_fraction[group_mask]

            valid = np.isfinite(group_fraction).all(axis=1)
            valid_data = group_data.loc[valid].copy()
            data = group_fraction[valid]

            if len(data) < 2:
                continue

            ellipse = ellipse_points(
                data[:, 0],
                data[:, 1],
                n_std=1.0,
            )

            color = state.name_to_color.get(source_group, "gray")

            if ellipse is not None:
                a_values = ellipse[:, 0]
                b_values = ellipse[:, 1]
                c_values = 1 - a_values - b_values

                fig.add_trace(go.Scatterternary(
                    a=a_values,
                    b=b_values,
                    c=c_values,
                    mode="lines",
                    line=dict(color=color),
                    fill="toself",
                    opacity=0.35,
                    name=f"{source_group} Source",
                ))

            if show_source_points:
                point_labels = (
                    valid_data["Sample"]
                    if "Sample" in valid_data.columns
                    else valid_data["Name"]
                    if "Name" in valid_data.columns
                    else None
                )

                fig.add_trace(go.Scatterternary(
                    a=data[:, 0],
                    b=data[:, 1],
                    c=data[:, 2],
                    mode="markers",
                    name=f"{source_group} Source Points",
                    text=point_labels,
                    hovertemplate=(
                        "Source sample: %{text}<extra></extra>"
                    ),
                    marker=dict(
                        size=6,
                        color=color,
                        opacity=0.8,
                        line=dict(width=1, color="black"),
                    ),
                    showlegend=False,
                ))

    # Study samples.
    if (
        study_fraction is not None
        and not study.empty
        and GROUP_COL in study.columns
    ):
        for study_group in study[GROUP_COL].dropna().unique():
            group_mask = (
                study[GROUP_COL].astype(str)
                == str(study_group)
            ).to_numpy()

            group_data = study.loc[group_mask].copy()
            group_fraction = study_fraction[group_mask]

            valid = np.isfinite(group_fraction).all(axis=1)
            data = group_fraction[valid]
            valid_data = group_data.loc[valid]

            if len(data) == 0:
                continue

            color = state.name_to_color.get(study_group, "gray")

            point_labels = (
                valid_data["Name"].to_numpy()
                if "Name" in valid_data.columns
                else valid_data["Sample"].to_numpy()
                if "Sample" in valid_data.columns
                else [str(study_group)] * len(data)
            )

            fig.add_trace(go.Scatterternary(
                a=data[:, 0],
                b=data[:, 1],
                c=data[:, 2],
                mode="markers+text" if show_labels else "markers",
                name=f"Study: {study_group}",
                text=point_labels,
                textposition="top center",
                hovertemplate="Name: %{text}<extra></extra>",
                marker=dict(size=8, color=color),
                showlegend=True,
            ))

    fig.update_layout(
        ternary=dict(
            sum=1,
            aaxis=dict(title=a_col),
            baxis=dict(title=b_col),
            caxis=dict(title=c_col),
        ),
        title=dict(
            text=f"Ternary Plot: {a_col}, {b_col}, {c_col}",
            x=0.5,
            xanchor="center",
        ),
        legend=dict(
            x=1.02,
            y=1,
            xanchor="left",
            yanchor="top",
        ),
        height=550,
    )

    return fig


def render_ternary_ui(
    host_out,
    show_labels_getter=lambda: False,
    show_source_points_getter=lambda: True,
):
    """Render the ternary plot section."""
    plot_elements = ["Rb", "Sr", "Zr", "Ba", "Nb", "Y", "Th", "U"]

    ternary_elements = [
        column
        for column in plot_elements
        if column in state.srcs.columns
        and column in state.study.columns
    ]

    with host_out:
        display(HTML(
            "<h3>Ternary Plot</h3>"
            "<p style='color:grey;font-size:12px;'>"
            "Use the shared controls above to show labels and source points."
            "</p>"
        ))

        if len(ternary_elements) < 3:
            print(
                "⚠️ Fewer than three common geochemical columns are "
                "available in srcs and study."
            )
            return

        a_default = "Rb" if "Rb" in ternary_elements else ternary_elements[0]
        b_default = "Sr" if "Sr" in ternary_elements else ternary_elements[1]
        c_default = "Zr" if "Zr" in ternary_elements else ternary_elements[2]

        ternary_a_menu = widgets.Dropdown(
            options=ternary_elements,
            value=a_default,
            description="A axis:",
        )
        ternary_b_menu = widgets.Dropdown(
            options=ternary_elements,
            value=b_default,
            description="B axis:",
        )
        ternary_c_menu = widgets.Dropdown(
            options=ternary_elements,
            value=c_default,
            description="C axis:",
        )

        display(widgets.HBox([
            ternary_a_menu,
            ternary_b_menu,
            ternary_c_menu,
        ]))

        ternary_output = widgets.Output()

        def update_ternary(change=None):
            with ternary_output:
                ternary_output.clear_output(wait=True)
                display(build_ternary_plot(
                    ternary_a_menu.value,
                    ternary_b_menu.value,
                    ternary_c_menu.value,
                    show_labels=show_labels_getter(),
                    show_source_points=show_source_points_getter(),
                ))

        ternary_a_menu.observe(update_ternary, names="value")
        ternary_b_menu.observe(update_ternary, names="value")
        ternary_c_menu.observe(update_ternary, names="value")

        display(ternary_output)
        update_ternary()


def render_analysis(host_out):
    """Render Tab 3 with shared controls for both graphs."""
    with host_out:
        host_out.clear_output(wait=True)

        display(widgets.HTML(
            "<h3>Step 3 — Analysis updates automatically</h3>"
            "<p>The summary, biplot, and ternary plot reflect the "
            "sources selected on the map.</p>"
        ))

        labels_button = widgets.ToggleButton(
            value=False,
            description="Show Labels",
            icon="tag",
            layout=widgets.Layout(width="150px"),
        )

        source_points_button = widgets.ToggleButton(
            value=False,
            description="Show Source Points",
            icon="bullseye",
            layout=widgets.Layout(width="190px"),
        )

        display(widgets.HBox([
            labels_button,
            source_points_button,
        ]))

        biplot_out = widgets.Output(
            layout=widgets.Layout(
                width="50%",
                flex="1 1 50%",
            )
        )

        ternary_out = widgets.Output(
            layout=widgets.Layout(
                width="50%",
                flex="1 1 50%",
            )
        )

        display(widgets.HBox(
            [biplot_out, ternary_out],
            layout=widgets.Layout(
                width="100%",
                display="flex",
                flex_flow="row nowrap",
                align_items="flex-start",
                gap="12px",
            ),
        ))


    apply_selection()
    build_color_map()

    def show_labels():
        return labels_button.value

    def show_source_points():
        return source_points_button.value

    def refresh_plots(change=None):
        biplot_out.clear_output(wait=True)
        ternary_out.clear_output(wait=True)

        render_biplot_ui(
            biplot_out,
            show_labels_getter=show_labels,
            show_source_points_getter=show_source_points,
        )

        render_ternary_ui(
            ternary_out,
            show_labels_getter=show_labels,
            show_source_points_getter=show_source_points,
        )

    labels_button.observe(refresh_plots, names="value")
    source_points_button.observe(refresh_plots, names="value")

    refresh_plots()

In [8]:
# CELL 8 - DATA LOADERS

def _normal_column_name(value):
    """Normalize a column name for case-insensitive matching."""
    return re.sub(r"[\s_\-]+", "", str(value).strip().lower())


def _standardize_table_columns(table, table_type):
    """Rename common alternative column names to the notebook's canonical names."""
    if not isinstance(table, pd.DataFrame):
        return table

    table = table.copy()
    table.columns = [str(column).strip() for column in table.columns]

    normalized = {
        _normal_column_name(column): column
        for column in table.columns
    }

    aliases = {
        "Group": ["group", "sourcegroup", "sourcename", "application"],
        "Sample": ["sample", "source", "name"],
    }

    if table_type == "locations":
        aliases = {
            "Group": ["group", "sourcegroup", "sourcename", "application"],
            "Name": ["name", "source", "sample"],
            "Lat": ["lat", "latitude"],
            "Long": ["long", "longitude", "lon"],
        }
    elif table_type in {"sources", "study"}:
        aliases = {
            "Group": ["group", "sourcegroup", "sourcename", "application"],
            "Sample": ["sample", "source", "name"],
        }

    rename = {}

    for canonical, possible_names in aliases.items():
        if canonical in table.columns:
            continue

        for possible_name in possible_names:
            original = normalized.get(_normal_column_name(possible_name))

            if original is not None:
                rename[original] = canonical
                break

    table = table.rename(columns=rename)

    # Remove duplicate columns created by alias conversion.
    table = table.loc[:, ~table.columns.duplicated(keep="first")]

    return table


def _load_table_from_url(url):
    url = (url or "").strip()

    if not url:
        raise ValueError("Please provide a URL.")

    if "docs.google.com/spreadsheets" in url:
        url = gsheet_to_csv_url(url)

    return read_data_table(url, url)


def _display_table_loader(number, title, raw_attribute, help_text):
    upload = widgets.FileUpload(
        accept=".csv,.tsv,.tab,.txt,.xlsx,.xls",
        multiple=False,
        description="Browse...",
        layout=widgets.Layout(width="130px"),
    )

    load_button = widgets.Button(
        description="Load",
        icon="upload",
        button_style="primary",
        layout=widgets.Layout(width="90px"),
    )

    url_field = widgets.Text(
        placeholder="or provide URL to online table or Google Sheets",
        layout=widgets.Layout(width="100%"),
    )

    url_button = widgets.Button(
        description="Load URL",
        icon="download",
        layout=widgets.Layout(width="110px"),
    )

    output = widgets.Output(
        layout=widgets.Layout(width="100%")
    )

    table_type = {
        "srcs": "sources",
        "srcs_locs": "locations",
        "study": "study",
    }[raw_attribute]

    def save_table(table, source):
        if not isinstance(table, pd.DataFrame) or table.empty:
            raise ValueError("The table is empty.")

        table = _standardize_table_columns(table, table_type)
        setattr(_raw, raw_attribute, table)

        with output:
            output.clear_output(wait=True)
            print(
                f"✅ Loaded from {source}: "
                f"{len(table)} rows × {len(table.columns)} columns"
            )
            print(f"Columns: {', '.join(map(str, table.columns))}")
            display(table.head())

    def on_load_click(_):
        uploaded = _first_upload(upload)

        if uploaded is None:
            with output:
                output.clear_output(wait=True)
                print("⚠️ Browse to a file first, then click Load.")
            return

        try:
            content = uploaded.get("content")
            filename = uploaded.get("name", "local file")
            table = read_data_table(content, filename)
            save_table(table, filename)
        except Exception as exc:
            with output:
                output.clear_output(wait=True)
                print(f"❌ Could not load {title}: {exc}")

    def on_url_click(_):
        try:
            table = _load_table_from_url(url_field.value)
            save_table(table, "online URL")
        except Exception as exc:
            with output:
                output.clear_output(wait=True)
                print(f"❌ Could not load {title}: {exc}")

    load_button.on_click(on_load_click)
    url_button.on_click(on_url_click)

    heading = widgets.HTML(
        value=(
            f"<b>{number}. {title}</b> "
            f"<span title='{help_text}' "
            "style='cursor:help;color:#555;font-size:16px;'>ⓘ</span>"
        ),
        layout=widgets.Layout(width="100%"),
    )

    file_controls = widgets.HBox(
        [upload, load_button],
        layout=widgets.Layout(
            width="100%",
            flex_flow="row wrap",
            align_items="center",
        ),
    )

    url_controls = widgets.HBox(
        [url_field, url_button],
        layout=widgets.Layout(
            width="100%",
            flex_flow="row wrap",
            align_items="center",
        ),
    )

    return widgets.VBox(
        [
            heading,
            file_controls,
            url_controls,
            output,
            widgets.HTML("<hr>"),
        ],
        layout=widgets.Layout(width="100%"),
    )


def build_data_upload_ui(host_out):
    source_loader = _display_table_loader(
        1,
        "Obsidian Source Chemistry",
        "srcs",
        "Source chemistry table.",
    )

    location_loader = _display_table_loader(
        2,
        "Obsidian Source Location Table",
        "srcs_locs",
        "Source locations with Group, Lat, and Long columns.",
    )

    study_loader = _display_table_loader(
        3,
        "Obsidian Study Chemistry",
        "study",
        "Study-sample chemistry table.",
    )

    continue_output = widgets.Output(
        layout=widgets.Layout(width="100%")
    )

    continue_button = widgets.Button(
        description="Load & Continue ▶",
        button_style="primary",
        icon="arrow-right",
    )

    def on_continue(_):
        with continue_output:
            continue_output.clear_output(wait=True)

            required_tables = [
                ("srcs", "Obsidian Source Chemistry"),
                ("srcs_locs", "Obsidian Source Location Table"),
                ("study", "Obsidian Study Chemistry"),
            ]

            missing = [
                label
                for attribute, label in required_tables
                if not isinstance(getattr(_raw, attribute), pd.DataFrame)
                or getattr(_raw, attribute).empty
            ]

            if missing:
                print("⚠️ Please load:")
                for label in missing:
                    print(f"   • {label}")
                return

            try:
                clean_and_validate(
                    _raw.srcs,
                    _raw.srcs_locs,
                    _raw.study,
                )
                build_map_selection_ui(tab2_out)
                tabs.selected_index = 1
                print("✅ All tables loaded successfully.")
            except Exception as exc:
                print(f"❌ Could not continue: {exc}")

    continue_button.on_click(on_continue)

    app = widgets.VBox(
        [
            widgets.HTML(
                value=(
                    "<h3>Step 1 — Data Upload</h3>"
                    "<p>Upload a table with <b>Browse...</b>, then click "
                    "<b>Load</b>, or provide a URL to an online table or "
                    "Google Sheet and click <b>Load URL</b>.</p>"
                )
            ),
            source_loader,
            location_loader,
            study_loader,
            continue_button,
            continue_output,
        ],
        layout=widgets.Layout(width="100%"),
    )

    host_out.clear_output(wait=True)
    display(app)

In [9]:
# CELL 9 - APPLICATION TABS
# Defined last so every callback dependency above is guaranteed to exist
# before Voilà/Binder renders the interactive widgets.

tab1_out = widgets.Output()
tab2_out = widgets.Output()
tab3_out = widgets.Output()

tabs = widgets.Tab(children=[tab1_out, tab2_out, tab3_out])
tabs.set_title(0, "1. Data Upload")
tabs.set_title(1, "2. Map Selection")
tabs.set_title(2, "3. Biplot & Ternary")

with tab2_out:
    display(widgets.HTML(
        "<p style='color:grey;'>Complete Tab 1 (Data Upload) and click "
        "\u201cLoad & Continue\u201d to display the source map here.</p>"
    ))
with tab3_out:
    display(widgets.HTML(
        "<p style='color:grey;'>Lasso-select sources on Tab 2 (Map Selection) and click "
        "\u201cGet Selection\u201d to display the biplot and ternary plot here.</p>"
    ))

build_data_upload_ui(tab1_out)

display(tabs)


### The analysis notebook has concluded.
In your data exploration, mouse over the points in the biplot and ternary plot to view the unique ID numbers of those samples and note their relationship to ellipses for known obsidian sources.
